In [1]:
import sys
from pathlib import Path
import torch
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.functional as F

# parent directory to path for imports
sys.path.append(str(Path.cwd().parent))

#  import utility functions from existing modules
from train_cvae import load_data, CVAE, one_hot
from ae_utils import _construct_latent_vectors_list, _reconstuct_from_latent_vectors_list
from ae_utils import _plot_latent_space, _save_vectors_to_csv

In [ ]:
# config
root = Path.cwd().parent
dataset_type = 'mnist'  # Choose: 'mnist' or 'adima'

# model config
if dataset_type == 'mnist':
    checkpoint_path = root / 'src/training/runs/cvaev2203_2/vae_1.pth' ##### CHANGE THIS PATH TO A REAL ONE ######
    input_dim = 784  # 28x28
    latent_dim = 10
    data_csv = None  # MNIST is from torchvision

print(f"dataset: {dataset_type}")
print(f"checkpoint: {checkpoint_path}")
print(f"input dim: {input_dim}, latent dim: {latent_dim}")


X_train_tensor, X_valid_tensor, X, _, y_train = load_data(
    data_csv=data_csv, 
    dataset=dataset_type, 
    single_class=None
)

dataset: mnist
checkpoint: /home/elsheikh/repos/src/training/runs/cvaev2203_2/vae_1.pth
input dim: 784, latent dim: 10
labels head(): [5 0 4 1 9]


In [3]:
if dataset_type == 'mnist':
    num_classes = 10
    # y_train is already an integer numpy array (0-9)
    labels_tensor = torch.tensor(y_train, dtype=torch.long)

one_hot_labels = F.one_hot(labels_tensor, num_classes=num_classes).float()
y_train_one_hot = one_hot_labels

In [4]:
# encode the training dataset
ae_type = 3  # 1 for VAE, 0 for AE, 3 for cvae
print("type(y_train):", type(y_train))
print("y_train.shape:", y_train.shape)
print("first y_train element:", y_train[0])

latent_vectors_all = _construct_latent_vectors_list(
    checkpoint_path=checkpoint_path,
    X_training_tensor=X_train_tensor,
    input_dim=input_dim,
    latent_dim=latent_dim,
    ae_type=ae_type,
    c=y_train_one_hot,
    dataset=dataset_type,
    class_size=num_classes
)

print(f"latent vectors shape: {latent_vectors_all.shape}")

type(y_train): <class 'numpy.ndarray'>
y_train.shape: (54000,)
first y_train element: 1
loading model from /home/elsheikh/repos/src/training/runs/cvaev2203_2/vae_1.pth


FileNotFoundError: [Errno 2] No such file or directory: '/home/elsheikh/repos/src/training/runs/cvaev2203_2/vae_1.pth'

In [ ]:
# reconstruct
reconstructed_samples = _reconstuct_from_latent_vectors_list(
    checkpoint_path=checkpoint_path,
    latent_vectors=latent_vectors_all,
    input_dim=input_dim,
    latent_dim=latent_dim,
    ae_type=ae_type,
    c=y_train_one_hot
)

In [ ]:
#%matplotlib inline
%matplotlib widget
# plot the full latent space
plt.figure(figsize=(10, 8))
_plot_latent_space(latent_vectors_all, y_train, show=True)